# Labeled-only vs HDBSCAN mcs4: OOF neighbors and embeddings

Same EVA02-L-14-336 recipe, two identity-disjoint 5-fold OOF runs:

| Run | Train labels | Path |
| --- | --- | --- |
| labeled-only | original `train.csv` | `runs/cv/eva02_trial23` |
| + mcs4 pseudo | orig train + HDBSCAN test clusters (`min_cluster_size=4`) | `runs/cv/pseudo_iter001_mcs4` |

Validation is original train identities only. Query / gallery CSVs and `image_id` order match fold-for-fold, so a neighbor list from one model is comparable to the other. Only the 256-D vectors change.

HDBSCAN test clustering was an experiment. Serving uses the labeled-only trial23 full-retrain, not the mcs4 weights.

Protocol is the usual H7 junk mask: drop same `vehicle_id` and `camera_id`, keep other identities on the query camera. Metrics are full-gallery mAP (checkpoint / Optuna) and Rank-1.


In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
from sklearn.decomposition import PCA

from dataset.images import crop_bbox, image_path
from postproc.retrieval import normalize

plt.rcParams.update(
    {
        "figure.figsize": (10, 4),
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.4f}".format)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "train.csv").is_file())
LABELED = ROOT / "runs" / "cv" / "eva02_trial23"
MCS4 = ROOT / "runs" / "cv" / "pseudo_iter001_mcs4"
IMAGES = ROOT / "data" / "images"
CONTEXT = 6.201741080661072
FIG_DIR = ROOT / "notebooks" / "eva02" / "oof_compare_figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT", ROOT)
print("labeled", LABELED)
print("mcs4", MCS4)

/data/projects/ryzhichkin/ReID/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT /data/projects/ryzhichkin/ReID
labeled /data/projects/ryzhichkin/ReID/runs/cv/eva02_trial23
mcs4 /data/projects/ryzhichkin/ReID/runs/cv/pseudo_iter001_mcs4


/data/projects/ryzhichkin/ReID/.venv/lib/python3.11/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error timed out>
  data = fetch_version_info()


## Load both OOF packs


In [2]:
def load_pack(directory):
    oof = pd.read_csv(directory / "oof.csv", dtype={"image_id": str})
    query = pd.read_csv(directory / "query.csv", dtype={"image_id": str})
    gallery = pd.read_csv(directory / "gallery.csv", dtype={"image_id": str})
    emb = normalize(np.load(directory / "embeddings.npy"))
    lookup = {image_id: i for i, image_id in enumerate(oof.image_id)}
    qe = emb[[lookup[i] for i in query.image_id]]
    ge = emb[[lookup[i] for i in gallery.image_id]]
    return {"oof": oof, "query": query, "gallery": gallery, "emb": emb, "qe": qe, "ge": ge, "lookup": lookup}


def query_rows(pack, fold):
    q, g, qe, ge = pack["query"], pack["gallery"], pack["qe"], pack["ge"]
    sim = qe @ ge.T
    gvid = g.vehicle_id.to_numpy()
    gcam = g.camera_id.to_numpy()
    gid = g.image_id.to_numpy()
    rows = []
    for i in range(len(q)):
        rec = q.iloc[i]
        keep = ~((gvid == rec.vehicle_id) & (gcam == rec.camera_id))
        order = np.argsort(-sim[i], kind="stable")
        order = order[keep[order]]
        relevant = gvid[order] == rec.vehicle_id
        found = np.flatnonzero(relevant)
        if len(found) == 0:
            continue
        first = int(found[0])
        rows.append(
            {
                "fold": fold,
                "q_index": i,
                "image_id": rec.image_id,
                "vehicle_id": int(rec.vehicle_id),
                "ap": float(np.mean(np.arange(1, len(found) + 1) / (found + 1))),
                "rank1_ok": bool(relevant[0]),
                "pos_rank": int(first + 1),
                "n_pos": int(relevant.sum()),
                "r1_image": str(gid[order[0]]),
                "r1_vid": int(gvid[order[0]]),
                "r1_cos": float(sim[i, order[0]]),
                "pos_image": str(gid[order[first]]),
                "pos_cos": float(sim[i, order[first]]),
                "top5": tuple(str(x) for x in gid[order[:5]]),
                "top10": tuple(str(x) for x in gid[order[:10]]),
            }
        )
    return pd.DataFrame(rows)


LABELED_FOLDS = {fold: load_pack(LABELED / f"fold{fold}" / "val") for fold in range(5)}
MCS4_FOLDS = {fold: load_pack(MCS4 / f"fold{fold}" / "val") for fold in range(5)}
for fold in range(5):
    old, new = LABELED_FOLDS[fold], MCS4_FOLDS[fold]
    assert list(old["query"].image_id) == list(new["query"].image_id)
    assert list(old["gallery"].image_id) == list(new["gallery"].image_id)
    assert list(old["oof"].image_id) == list(new["oof"].image_id)

old_tab = pd.concat([query_rows(LABELED_FOLDS[fold], fold) for fold in range(5)], ignore_index=True)
new_tab = pd.concat([query_rows(MCS4_FOLDS[fold], fold) for fold in range(5)], ignore_index=True)
CMP = old_tab.merge(new_tab, on="image_id", suffixes=("_lab", "_mcs"))
LABELED_METRICS = json.loads((LABELED / "cv_metrics.json").read_text())["metrics"]
MCS4_METRICS = json.loads((MCS4 / "cv_metrics.json").read_text())["metrics"]
print("queries", len(CMP), "folds aligned")
print(
    "labeled mAP",
    round(float(LABELED_METRICS["mAP"]["query_weighted_mean"]), 4),
    "mcs4",
    round(float(MCS4_METRICS["mAP"]["query_weighted_mean"]), 4),
)

queries 1541 folds aligned
labeled mAP 0.8467 mcs4 0.8565


## Per-query ranking

mcs4 is a net gain, not a uniform lift. A minority of queries move a lot; most stay put.


In [3]:
CMP["d_ap"] = CMP.ap_mcs - CMP.ap_lab
rescues = (~CMP.rank1_ok_lab) & CMP.rank1_ok_mcs
breaks = CMP.rank1_ok_lab & (~CMP.rank1_ok_mcs)
scoreboard = pd.DataFrame(
    {
        "labeled-only": [
            float(LABELED_METRICS["mAP"]["query_weighted_mean"]),
            float(LABELED_METRICS["mAP@10"]["query_weighted_mean"]),
            float(LABELED_METRICS["Rank-1"]["query_weighted_mean"]),
            float(CMP.pos_rank_lab.mean()),
        ],
        "mcs4": [
            float(MCS4_METRICS["mAP"]["query_weighted_mean"]),
            float(MCS4_METRICS["mAP@10"]["query_weighted_mean"]),
            float(MCS4_METRICS["Rank-1"]["query_weighted_mean"]),
            float(CMP.pos_rank_mcs.mean()),
        ],
    },
    index=pd.Index(["mAP", "mAP@10", "Rank-1", "mean first-positive rank"]),
)
scoreboard["delta"] = scoreboard["mcs4"] - scoreboard["labeled-only"]
display(scoreboard)
print(
    "Rank-1 both hit",
    int((CMP.rank1_ok_lab & CMP.rank1_ok_mcs).sum()),
    "rescue",
    int(rescues.sum()),
    "break",
    int(breaks.sum()),
    "both miss",
    int((~CMP.rank1_ok_lab & ~CMP.rank1_ok_mcs).sum()),
)
print(
    "AP up",
    int((CMP.d_ap > 1e-6).sum()),
    "down",
    int((CMP.d_ap < -1e-6).sum()),
    "unchanged",
    int((CMP.d_ap.abs() <= 1e-6).sum()),
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].scatter(CMP.ap_lab, CMP.ap_mcs, s=12, alpha=0.35, c="#4c78a8")
axes[0].plot([0, 1], [0, 1], color="black", lw=1)
axes[0].set_xlabel("labeled-only AP")
axes[0].set_ylabel("mcs4 AP")
axes[0].set_title("Per-query AP")
axes[1].hist(CMP.d_ap, bins=30, color="#54a24b", edgecolor="white")
axes[1].axvline(0, color="black")
axes[1].set_xlabel("Δ AP (mcs4 − labeled)")
axes[1].set_ylabel("queries")
axes[1].set_title("Most queries barely move")
plt.tight_layout()
fig.savefig(FIG_DIR / "ap_scatter.png", dpi=140, bbox_inches="tight")
plt.show()

,labeled-only,mcs4,delta
mAP,0.8467,0.8565,0.0098
mAP@10,0.8344,0.8456,0.0112
Rank-1,0.8423,0.8533,0.0110
mean first-positive rank,2.7047,2.1622,-0.5425


Rank-1 both hit 1232 rescue 83 break 66 both miss 160
AP up 336 down 278 unchanged 927


## Neighbor lists

Same query, same gallery. Overlap is the fraction of the labeled-only top-k that mcs4 still ranks in its top-k. Jaccard is that intersection over the union.


In [4]:
def overlap_at(old_lists, new_lists, k):
    hits = []
    jacc = []
    for a, b in zip(old_lists, new_lists, strict=True):
        sa, sb = set(a[:k]), set(b[:k])
        hits.append(len(sa & sb) / k)
        jacc.append(len(sa & sb) / len(sa | sb))
    return float(np.mean(hits)), float(np.mean(jacc))


neighbor_rows = []
for k, col in ((1, "r1_image"), (5, "top5"), (10, "top10")):
    if k == 1:
        ov = float((CMP.r1_image_lab == CMP.r1_image_mcs).mean())
        neighbor_rows.append({"k": 1, "same image": ov, "overlap@k": ov, "jaccard": ov})
    else:
        ov, jac = overlap_at(CMP[f"{col}_lab"], CMP[f"{col}_mcs"], k)
        neighbor_rows.append({"k": k, "same image": np.nan, "overlap@k": ov, "jaccard": jac})
neighbors = pd.DataFrame(neighbor_rows)
display(neighbors)

fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.bar(["top-1 same", "top-5 overlap", "top-10 overlap"], neighbors["overlap@k"], color="#4c78a8")
ax.set_ylim(0, 1)
ax.set_ylabel("fraction")
ax.set_title("How much of the labeled-only shortlist mcs4 keeps")
plt.tight_layout()
fig.savefig(FIG_DIR / "neighbor_overlap.png", dpi=140, bbox_inches="tight")
plt.show()

,k,same image,overlap@k,jaccard
0,1,0.5542,0.5542,0.5542
1,5,NaN,0.7324,0.6218
2,10,NaN,0.6655,0.5270


## Neighbor crops

Fold-0 examples. Each row is one query: labeled-only Rank-1 / first true positive, then the mcs4 Rank-1 / first true positive. Green border is a correct identity; red is a lookalike.


In [5]:
def show_crop(frame, image_id):
    row = frame.loc[frame.image_id == image_id].iloc[0]
    with Image.open(image_path(IMAGES, row.image_id)) as im:
        return crop_bbox(im.convert("RGB"), [row.x, row.y, row.w, row.h], CONTEXT)


def panel(ax, crop, title, edge):
    ax.imshow(crop)
    ax.set_title(title, fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2)
        spine.set_color(edge)


def pick(mask, n):
    part = CMP.loc[mask].copy()
    part["gain"] = part.d_ap.abs()
    return part.sort_values("gain", ascending=False).head(n)


cases = pd.concat(
    [
        pick((CMP.fold_lab == 0) & rescues, 2).assign(kind="rescue"),
        pick((CMP.fold_lab == 0) & breaks, 2).assign(kind="break"),
    ]
)
fig, axes = plt.subplots(len(cases), 5, figsize=(13.5, 2.45 * len(cases)))
for r, rec in enumerate(cases.to_dict("records")):
    pack = LABELED_FOLDS[int(rec["fold_lab"])]
    qf, gf = pack["query"], pack["gallery"]
    lab_ok = rec["rank1_ok_lab"]
    mcs_ok = rec["rank1_ok_mcs"]
    panel(axes[r, 0], show_crop(qf, rec["image_id"]), f"Q id {rec['vehicle_id_lab']}\n{rec['kind']}", "white")
    panel(
        axes[r, 1],
        show_crop(gf, rec["r1_image_lab"]),
        f"labeled Rank-1\nid {rec['r1_vid_lab']}  AP {rec['ap_lab']:.2f}",
        "#54a24b" if lab_ok else "#e45756",
    )
    panel(
        axes[r, 2],
        show_crop(gf, rec["pos_image_lab"]),
        f"labeled first +\nrank {rec['pos_rank_lab']}",
        "#4c78a8",
    )
    panel(
        axes[r, 3],
        show_crop(gf, rec["r1_image_mcs"]),
        f"mcs4 Rank-1\nid {rec['r1_vid_mcs']}  AP {rec['ap_mcs']:.2f}",
        "#54a24b" if mcs_ok else "#e45756",
    )
    panel(
        axes[r, 4],
        show_crop(gf, rec["pos_image_mcs"]),
        f"mcs4 first +\nrank {rec['pos_rank_mcs']}",
        "#4c78a8",
    )
fig.suptitle("Fold-0 OOF: labeled-only vs mcs4 neighbors", y=1.01)
plt.tight_layout()
fig.savefig(FIG_DIR / "neighbors.png", dpi=120, bbox_inches="tight")
plt.show()
display(
    cases[
        [
            "kind",
            "vehicle_id_lab",
            "ap_lab",
            "ap_mcs",
            "pos_rank_lab",
            "pos_rank_mcs",
            "r1_vid_lab",
            "r1_vid_mcs",
        ]
    ]
)

,kind,vehicle_id_lab,ap_lab,ap_mcs,pos_rank_lab,pos_rank_mcs,r1_vid_lab,r1_vid_mcs
182,rescue,485,0.1073,0.8333,15,1,743,485
261,rescue,590,0.0607,0.5882,19,1,1005,590
176,break,1321,1.0000,0.1403,1,10,1321,620
6,break,513,1.0000,0.1429,1,7,513,115


## Embedding geometry

Raw cosine between the two models on the **same image** is near 0: the axes are rotated. After an orthogonal Procrustes map (labeled → mcs4, per fold, no scaling) the vectors agree. Linear CKA on the centered Gram matrices says the same thing without choosing an axis.

Intra-id cosine is all same-identity pairs in the fold OOF (including same camera). Retrieval positives below are the cross-camera query–gallery hits after the junk mask.


In [6]:
def orthogonal_procrustes(source, target):
    u, _, vt = np.linalg.svd(source.T @ target, full_matrices=False)
    return normalize(source @ (u @ vt))


def linear_cka(left, right):
    left = left - left.mean(0)
    right = right - right.mean(0)
    gram_l = left @ left.T
    gram_r = right @ right.T
    hsic = float(np.sum(gram_l * gram_r))
    return hsic / float(np.sqrt(np.sum(gram_l**2) * np.sum(gram_r**2)))


def pair_cosines(pack, n_neg=20, seed=0):
    q, g, qe, ge = pack["query"], pack["gallery"], pack["qe"], pack["ge"]
    sim = qe @ ge.T
    gvid = g.vehicle_id.to_numpy()
    gcam = g.camera_id.to_numpy()
    rng = np.random.default_rng(seed)
    pos, neg = [], []
    for i in range(len(q)):
        rec = q.iloc[i]
        keep = ~((gvid == rec.vehicle_id) & (gcam == rec.camera_id))
        pos.extend(sim[i, keep & (gvid == rec.vehicle_id)].tolist())
        other = np.flatnonzero(keep & (gvid != rec.vehicle_id))
        take = other if len(other) <= n_neg else rng.choice(other, n_neg, replace=False)
        neg.extend(sim[i, take].tolist())
    return np.asarray(pos), np.asarray(neg)


def compactness(pack):
    oof, emb = pack["oof"], pack["emb"]
    intra = []
    for _, grp in oof.reset_index(drop=True).groupby("vehicle_id"):
        idx = grp.index.to_numpy()
        if len(idx) < 2:
            continue
        vec = emb[idx]
        gram = vec @ vec.T
        intra.append(float(gram[np.triu_indices(len(idx), 1)].mean()))
    return float(np.mean(intra))


raw_cos, aligned_cos, cka_vals = [], [], []
for fold in range(5):
    a = LABELED_FOLDS[fold]["emb"]
    b = MCS4_FOLDS[fold]["emb"]
    raw_cos.append(np.sum(a * b, axis=1))
    aligned_cos.append(np.sum(orthogonal_procrustes(a, b) * b, axis=1))
    cka_vals.append(linear_cka(a, b))

raw = np.concatenate(raw_cos)
aligned = np.concatenate(aligned_cos)
pos_lab, neg_lab = pair_cosines(LABELED_FOLDS[0])
pos_mcs, neg_mcs = pair_cosines(MCS4_FOLDS[0])
pair = pd.Series(
    {
        "same-image cosine (raw)": float(raw.mean()),
        "same-image cosine (Procrustes)": float(aligned.mean()),
        "linear CKA": float(np.mean(cka_vals)),
    },
    name="labeled vs mcs4",
)
geom = pd.DataFrame(
    {
        "labeled-only": [
            compactness(LABELED_FOLDS[0]),
            float(pos_lab.mean()),
            float(neg_lab.mean()),
        ],
        "mcs4": [
            compactness(MCS4_FOLDS[0]),
            float(pos_mcs.mean()),
            float(neg_mcs.mean()),
        ],
    },
    index=pd.Index(["fold-0 intra-id cosine", "fold-0 cross-cam positive", "fold-0 sampled negative"]),
)
display(pair.to_frame())
display(geom)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.0))
axes[0].hist(raw, bins=40, histtype="step", lw=1.8, label="raw")
axes[0].hist(aligned, bins=40, histtype="step", lw=1.8, label="Procrustes")
axes[0].set_xlabel("cosine(labeled, mcs4) on the same image")
axes[0].set_ylabel("OOF images")
axes[0].set_title("Axes differ; neighborhoods mostly agree")
axes[0].legend()
parts = [pos_lab, pos_mcs, neg_lab, neg_mcs]
axes[1].boxplot(parts, tick_labels=["lab +", "mcs4 +", "lab −", "mcs4 −"], showfliers=False)
axes[1].set_ylabel("query–gallery cosine")
axes[1].set_title("Fold 0: positives tighten, negatives stay near 0")
plt.tight_layout()
fig.savefig(FIG_DIR / "embed_align.png", dpi=140, bbox_inches="tight")
plt.show()

,labeled vs mcs4
same-image cosine (raw),-0.0045
same-image cosine (Procrustes),0.8282
linear CKA,0.7640


,labeled-only,mcs4
fold-0 intra-id cosine,0.7626,0.8245
fold-0 cross-cam positive,0.6755,0.7643
fold-0 sampled negative,0.0027,0.0043


## Fold-0 PCA after alignment

Orthogonal Procrustes maps the labeled-only vectors onto the mcs4 axes. PCA is fit on mcs4 fold-0, then both clouds are plotted. Twelve identities, same colors in both panels.


In [7]:
fold0_lab = LABELED_FOLDS[0]
fold0_mcs = MCS4_FOLDS[0]
aligned = orthogonal_procrustes(fold0_lab["emb"], fold0_mcs["emb"])
pca = PCA(n_components=2, random_state=0).fit(fold0_mcs["emb"])
xy_mcs = pca.transform(fold0_mcs["emb"])
xy_lab = pca.transform(aligned)
oof = fold0_mcs["oof"]
rng = np.random.default_rng(0)
ids = rng.choice(oof.vehicle_id.unique(), size=12, replace=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, xy, title in (
    (axes[0], xy_lab, "labeled-only (Procrustes → mcs4)"),
    (axes[1], xy_mcs, "mcs4"),
):
    ax.scatter(xy[:, 0], xy[:, 1], s=6, c="#d0d0d0", alpha=0.35)
    palette = plt.cm.tab20(np.linspace(0, 1, len(ids), endpoint=False))
    for vid, color in zip(ids, palette, strict=True):
        mask = oof.vehicle_id.to_numpy() == vid
        ax.scatter(xy[mask, 0], xy[mask, 1], s=18, color=color, label=str(int(vid)))
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
axes[1].legend(ncols=2, fontsize=8, frameon=False, loc="best")
plt.tight_layout()
fig.savefig(FIG_DIR / "pca.png", dpi=140, bbox_inches="tight")
plt.show()

## Takeaways

- Query-weighted OOF: labeled-only **mAP 0.847 / mAP@10 0.834 / Rank-1 0.842** vs mcs4 **0.857 / 0.846 / 0.853**. Mean first-positive rank 2.70 → 2.16.
- Rank-1 is not a one-way rescue: **83** queries flip miss→hit, **66** flip hit→miss, **1232** stay correct. AP rises on 336 queries and falls on 278; 927 are unchanged.
- Neighbor lists move more than the metric suggests. Only **55%** of queries keep the same Rank-1 image. Top-5 overlap is **0.73**, top-10 **0.67**.
- The two 256-D bases are rotated (raw same-image cosine ≈ 0). After orthogonal Procrustes they agree (**0.83**); linear CKA is **0.76**. Retrieval neighborhoods are related, not a rewrite of the space.
- Fold-0 same-identity cosine tightens (intra 0.76 → 0.82, cross-camera positives 0.68 → 0.76) while sampled negatives stay near 0 (0.003 → 0.004). Pseudo-labels pull views of the same vehicle together without collapsing lookalikes.

Serving uses the labeled-only trial23 full-retrain. Pseudo-labeling was tried and is not used. This notebook is OOF only.
